# Ammonia Toxicity Analysis

Un-ionized ammonia (NH₃-N) is the primary toxic form for shrimp.
The fraction depends on pH and temperature.

**Acute toxicity threshold:** 0.3 mg NH₃-N/L (Wickins, 1976; Boyd & Tucker, 1998)
**Chronic toxicity threshold:** 0.1 mg NH₃-N/L

This notebook explores how TAN, pH, and temperature interact to
determine NH₃-N exposure risk in intensive shrimp ponds.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.dpi'] = 120
from xoceania_sim import PondSimulator, SimConfig
from xoceania_sim.subsystems.nitrogen import nh3_fraction

## 1. NH₃ ionization fraction vs pH and temperature

In [ ]:
pH_range = np.linspace(6.5, 10.0, 100)
temps = [25, 28, 30, 32, 35]
colors = ['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd']

fig, ax = plt.subplots(figsize=(9, 5))
for T, color in zip(temps, colors):
    fracs = [nh3_fraction(pH, T) for pH in pH_range]
    ax.plot(pH_range, fracs, color=color, lw=2, label=f'{T}°C')

ax.set_xlabel('pH', fontsize=12)
ax.set_ylabel('NH₃ ionization fraction (α)', fontsize=12)
ax.set_title('Un-ionized NH₃ Fraction vs pH and Temperature', fontsize=12, fontweight='bold')
ax.legend(title='Temperature', fontsize=10)
ax.set_xlim(6.5, 10.0)
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Simulate TAN and NH₃ over 48 hours at different stocking densities

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
densities = [50, 100, 150, 200]
palette = ['#1f77b4', '#ff7f0e', '#d62728', '#9467bd']

for density, color in zip(densities, palette):
    cfg = SimConfig()
    cfg.shrimp.stocking_density_m2 = density
    sim = PondSimulator(cfg)
    result = sim.run(t_span=(0, 48))
    df = result.df
    t = df.index
    axes[0].plot(t, df['TAN'], color=color, lw=2, label=f'{density} PL/m²')
    axes[1].plot(t, df['NH3'], color=color, lw=2, label=f'{density} PL/m²')

axes[0].set_ylabel('TAN (mg N/L)')
axes[0].legend(fontsize=9, title='Stocking density')
axes[1].set_ylabel('NH₃-N (mg N/L)')
axes[1].axhline(0.1, color='orange', ls='--', lw=1.5, label='Chronic threshold 0.1')
axes[1].axhline(0.3, color='red', ls='--', lw=1.5, label='Acute threshold 0.3')
axes[1].legend(fontsize=9)
axes[1].set_xlabel('Time (h)')
axes[1].set_xticks(range(0, 49, 6))

for ax in axes:
    ax.grid(True, alpha=0.3)

fig.suptitle('TAN and NH₃-N Dynamics vs Stocking Density', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Risk map: NH₃ toxicity probability across pH × TAN space

In [ ]:
pH_vals = np.linspace(7.0, 10.0, 50)
TAN_vals = np.linspace(0.0, 5.0, 50)
T_water = 29.0  # typical Mekong Delta temperature

pH_grid, TAN_grid = np.meshgrid(pH_vals, TAN_vals)
NH3_grid = np.vectorize(lambda ph, tan: nh3_fraction(ph, T_water) * tan)(pH_grid, TAN_grid)

fig, ax = plt.subplots(figsize=(8, 6))
CS = ax.contourf(pH_grid, TAN_grid, NH3_grid, levels=20, cmap='RdYlGn_r')
plt.colorbar(CS, ax=ax, label='NH₃-N (mg N/L)')
ax.contour(pH_grid, TAN_grid, NH3_grid, levels=[0.1, 0.3],
           colors=['orange', 'red'], linewidths=2)
ax.set_xlabel('pH', fontsize=12)
ax.set_ylabel('TAN (mg N/L)', fontsize=12)
ax.set_title(f'NH₃-N Toxicity Risk Map at T={T_water}°C\n'
             f'Orange: chronic threshold (0.1), Red: acute threshold (0.3)',
             fontsize=11, fontweight='bold')
ax.grid(True, alpha=0.2, color='white')
plt.tight_layout()
plt.show()